# NYISO Extension: Zone J (NYC) and Zone K (Long Island)
Same pipeline as the PJM Western Hub model, applied to New York.

- **Primary Y:** daily on-peak day-ahead LBMP, NYISO Zone J (NYC), $/MWh
- **Secondary Y:** same for Zone K (Long Island)
- **On-peak:** weekdays, hours ending 8 to 23 Eastern, NERC holidays excluded (identical to PJM build)
- **X:** heating/cooling degree days (NYC for Zone J, Islip for Zone K), Henry Hub from the previous trading day, day-of-week and month fixed effects
- **Split:** train 2023 to 2025, test 2026 (chronological)

**Caveats:** (1) backtest uses observed weather, i.e. a perfect weather forecast; (2) Henry Hub understates NYC winter gas cost. Transco Zone 6 NY is the relevant gas price, but no free daily series exists (EIA licenses it from a vendor and publishes it only weekly). If you obtain one, save it as `transco_z6_ny.csv` (columns `date,gas`) and the notebook adds it as an alternative driver.

Run with **Runtime > Run all**. If any step fails, the notebook stops with a message instead of substituting values.

In [ ]:
# Step 1: install packages (Colab only)
import subprocess, sys, importlib.util
if importlib.util.find_spec("google.colab") is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gridstatusio", "xlrd"], check=False)

In [ ]:
# Step 2: settings
# Explain: one place for the study window, the train/test cutoff, and the weather stations.
import os, re, logging, warnings
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, USMemorialDay,
                                    USLaborDay, USThanksgivingDay, sunday_to_monday)

warnings.filterwarnings("ignore")
logging.getLogger("gridstatusio").setLevel(logging.WARNING)
pd.set_option("display.width", 160)

START = "2023-01-01"
END = (pd.Timestamp.today().normalize() - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
TEST_START = "2026-01-01"
STATIONS = {"J": ("New York City", 40.71, -74.01), "K": ("Islip (Long Island)", 40.79, -73.10)}
NY_CACHE = "nyiso_daily.csv"
NY_HOURLY_CACHE = "nyiso_hourly.csv"   # hourly Zone J/K prices, needed for the finance modules
# Modeled project sites (weather-based profiles, not actual plant output)
SOLAR_SITE = ("Long Island solar (Islip)", 40.79, -73.10)
WIND_SITE = ("Offshore wind, NY Bight south of Long Island", 40.25, -73.25)
PJM_CACHE = "pjm_daily.csv"   # reused from the PJM notebook if present in this session

class StopRun(Exception):
    """Raised when a step fails, so nothing downstream runs on bad or missing data."""

print("Data window:", START, "to", END)

In [ ]:
# Step 3: shared functions
# Explain: the same on-peak block logic as PJM: convert UTC to Eastern, tag weekday hours
# ending 8 to 23 (interval start 7:00 to 22:00), drop NERC holidays, then average by day.
class NERCHolidays(AbstractHolidayCalendar):
    rules = [
        Holiday("New Year", month=1, day=1, observance=sunday_to_monday),
        USMemorialDay,
        Holiday("Independence Day", month=7, day=4, observance=sunday_to_monday),
        USLaborDay,
        USThanksgivingDay,
        Holiday("Christmas", month=12, day=25, observance=sunday_to_monday),
    ]

HOLIDAYS = NERCHolidays().holidays(start="2022-12-01", end="2027-12-31")

def get_client():
    from gridstatusio import GridStatusClient
    try:
        from google.colab import userdata
        key = userdata.get("GRIDSTATUS_API_KEY")
    except Exception:
        key = os.environ.get("GRIDSTATUS_API_KEY")
    return GridStatusClient(api_key=key)

def daily_blocks(hourly, prefix):
    t = pd.to_datetime(hourly["interval_start_utc"], utc=True).dt.tz_convert("America/New_York")
    d = pd.DataFrame({"t": t.values, "lmp": hourly["lmp"].values})
    d["t"] = pd.to_datetime(d["t"], utc=True).dt.tz_convert("America/New_York")
    d["date"] = d["t"].dt.tz_localize(None).dt.normalize()
    d["onpeak"] = ((d["t"].dt.dayofweek < 5) & d["t"].dt.hour.between(7, 22)
                   & ~d["date"].isin(HOLIDAYS))
    out = d.groupby(["date", "onpeak"])["lmp"].mean().unstack()
    return out.rename(columns={True: f"{prefix}_on", False: f"{prefix}_off"})

def get_weather(lat, lon, prefix):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": lat, "longitude": lon, "start_date": START, "end_date": END,
        "daily": "temperature_2m_mean", "temperature_unit": "fahrenheit",
        "timezone": "America/New_York"}, timeout=60)
    if r.status_code != 200:
        raise StopRun(f"Open-Meteo returned {r.status_code}: {r.text[:200]}")
    js = r.json()["daily"]
    temp = pd.Series(js["temperature_2m_mean"], index=pd.to_datetime(js["time"]))
    w = pd.DataFrame({f"{prefix}_temp": temp,
                      f"{prefix}_hdd": (65 - temp).clip(lower=0),
                      f"{prefix}_cdd": (temp - 65).clip(lower=0)})
    w.index.name = "date"
    return w

def get_henry_hub():
    try:
        raw = pd.read_excel("https://www.eia.gov/dnav/ng/hist_xls/RNGWHHDd.xls",
                            sheet_name="Data 1", skiprows=2)
        g = raw.iloc[:, :2]
        g.columns = ["date", "gas"]
    except Exception as e:
        if os.path.exists("henry_hub.csv"):
            print("EIA download failed; using henry_hub.csv")
            g = pd.read_csv("henry_hub.csv")
        else:
            raise StopRun(f"Henry Hub download failed ({e}). Upload henry_hub.csv with columns date,gas and rerun.")
    g["date"] = pd.to_datetime(g["date"])
    return g.dropna().sort_values("date")

def attach_prev_day_gas(daily, gas, col):
    """Gas known before the DA market closes: last price traded strictly before the delivery day."""
    daily = daily.copy()
    daily["lookup"] = daily["date"] - pd.Timedelta(days=1)
    merged = pd.merge_asof(daily.sort_values("lookup"),
                           gas.rename(columns={"date": "gas_date", "gas": col}),
                           left_on="lookup", right_on="gas_date", direction="backward",
                           tolerance=pd.Timedelta(days=7))
    return merged.drop(columns=["lookup", "gas_date"]).sort_values("date").reset_index(drop=True)

def show(df):
    try:
        print(df.to_markdown(index=False))
    except Exception:
        print(df.to_string(index=False))

In [ ]:
# Step 4: find the NYISO dataset and the Zone J / Zone K location names
# Explain: dataset IDs and zone spellings come from the API itself, not from guesses.
client = get_client()
ids = [d["id"] for d in client.list_datasets(return_list=True)]
nyiso_lmp = [i for i in ids if i.startswith("nyiso") and ("lmp" in i or "lbmp" in i)]
print("NYISO price datasets available:\n ", "\n  ".join(nyiso_lmp))

da_candidates = [i for i in nyiso_lmp if "day_ahead" in i and "hourly" in i]
if not da_candidates:
    raise StopRun("No NYISO day-ahead hourly price dataset found. Check the list above and set DATASET manually.")
DATASET = "nyiso_lmp_day_ahead_hourly" if "nyiso_lmp_day_ahead_hourly" in da_candidates else da_candidates[0]
print("\nUsing dataset:", DATASET)

probe = client.get_dataset(dataset=DATASET, start="2026-11-02", end="2026-11-02 01:00", limit=5000)
locs = probe[["location", "location_type"]].drop_duplicates() if "location_type" in probe else probe[["location"]].drop_duplicates()
zone_j = [l for l in locs["location"] if re.fullmatch(r"N\.?\s?Y\.?\s?C\.?", str(l).strip(), re.I)]
zone_k = [l for l in locs["location"] if re.fullmatch(r"LONG\s?IL.*|LONG ISLAND", str(l).strip(), re.I)]
print("\nZone-type locations in the probe:")
print(locs[locs.get("location_type", pd.Series("", index=locs.index)).astype(str).str.upper().str.contains("ZONE")].to_string(index=False)
      if "location_type" in locs else locs.head(30).to_string(index=False))
if not zone_j or not zone_k:
    raise StopRun(f"Could not match Zone J/K names automatically (found J={zone_j}, K={zone_k}). "
                  "Read the list above and set ZONE_NAMES manually.")
ZONE_NAMES = {"J": zone_j[0], "K": zone_k[0]}
print("\nZone names:", ZONE_NAMES)

In [ ]:
# Step 5: build or load the daily NYISO dataset
# Explain: pull hourly DA prices for each zone only (filtered server-side to save API rows),
# build on-peak/off-peak blocks, join zone weather and previous-day Henry Hub.
if os.path.exists(NY_CACHE) and os.path.exists(NY_HOURLY_CACHE):
    ny = pd.read_csv(NY_CACHE, parse_dates=["date"])
    ny_hourly = pd.read_csv(NY_HOURLY_CACHE, parse_dates=["interval_start_utc"])
    ny_hourly["interval_start_utc"] = pd.to_datetime(ny_hourly["interval_start_utc"], utc=True)
    print("Loaded caches:", NY_CACHE, ny.shape, "|", NY_HOURLY_CACHE, ny_hourly.shape)
else:
    parts, hourly_cols = [], []
    for z, name in ZONE_NAMES.items():
        hourly = client.get_dataset(dataset=DATASET, start=START, end=END,
                                    filter_column="location", filter_value=name)
        if hourly.empty:
            raise StopRun(f"No rows returned for {name}.")
        print(f"Zone {z} ({name}): {len(hourly):,} hourly rows")
        parts.append(daily_blocks(hourly[["interval_start_utc", "lmp"]], z))
        h = hourly[["interval_start_utc", "lmp"]].copy()
        h["interval_start_utc"] = pd.to_datetime(h["interval_start_utc"], utc=True)
        hourly_cols.append(h.groupby("interval_start_utc")["lmp"].mean().rename(z))
    ny_hourly = pd.concat(hourly_cols, axis=1).reset_index()
    ny_hourly.to_csv(NY_HOURLY_CACHE, index=False)
    ny = parts[0].join(parts[1], how="outer")
    for z, (label, lat, lon) in STATIONS.items():
        ny = ny.join(get_weather(lat, lon, z), how="left")
    ny = ny.reset_index()
    ny = attach_prev_day_gas(ny, get_henry_hub(), "gas")
    if os.path.exists("transco_z6_ny.csv"):
        tz6 = pd.read_csv("transco_z6_ny.csv", parse_dates=["date"]).dropna().sort_values("date")
        ny = attach_prev_day_gas(ny, tz6, "gas_z6")
        print("Added Transco Z6 NY from transco_z6_ny.csv")
    ny.to_csv(NY_CACHE, index=False)
    print("Built and cached:", ny.shape)

print("\nMissing values:\n", ny.isna().sum())
ny.tail()

In [ ]:
# Step 6: model functions (identical method to the PJM build)
# Explain: every model forecasts a day using only information available before that day.
# Smoothing models use past prices; regressions use weather and previous-day gas.
def model_frame(daily, z, gas_col="gas"):
    df = daily[["date", f"{z}_on", f"{z}_temp", f"{z}_hdd", f"{z}_cdd", gas_col]].dropna().copy()
    df.columns = ["date", "da_on", "temp", "hdd", "cdd", "gas"]
    df = df.sort_values("date").reset_index(drop=True)
    df["dow"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["heat_rate"] = df["da_on"] / df["gas"]
    df["lag1"] = df["da_on"].shift(1)
    df["last_year"] = df["date"].map(df.set_index("date")["da_on"].rename(lambda d: d + pd.Timedelta(days=364)))
    return df.dropna(subset=["lag1"]).reset_index(drop=True)

FORMULAS = {
    "Regression: weather + gas + FE": "da_on ~ hdd + cdd + gas + C(dow) + C(month)",
    "Regression: + squared weather": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + C(dow) + C(month)",
    "Regression: + squared weather + lag": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + lag1 + C(dow) + C(month)",
}

def run_models(df):
    y = df["da_on"]
    tr = df["date"] < TEST_START
    train, test = df[tr], df[~tr]
    preds = pd.DataFrame(index=df.index)
    preds["Naive: previous day"] = df["lag1"]
    preds["Naive: same day last year"] = df["last_year"]
    for w in [2, 5, 10, 20, 60]:
        preds[f"Moving average {w}d"] = y.shift(1).rolling(w).mean()
    for w in [5, 20]:
        wt = np.arange(1, w + 1)
        preds[f"Weighted MA {w}d"] = y.shift(1).rolling(w).apply(lambda x: np.dot(x, wt) / wt.sum(), raw=True)
    alpha = min(np.arange(0.05, 1.0, 0.05),
                key=lambda a: (y - y.ewm(alpha=a, adjust=False).mean().shift(1))[tr].abs().mean())
    preds[f"Exp. smoothing (alpha={alpha:.2f})"] = y.ewm(alpha=alpha, adjust=False).mean().shift(1)
    fits = {}
    for name, f in FORMULAS.items():
        fits[name] = smf.ols(f, data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        preds.loc[test.index, name] = fits[name].predict(test)
    hr = smf.ols("heat_rate ~ hdd + I(hdd**2) + cdd + I(cdd**2) + C(dow) + C(month)",
                 data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    preds.loc[test.index, "Heat-rate model x gas"] = hr.predict(test) * test["gas"]
    try:
        def exog(d):
            X = d[["hdd", "cdd", "gas"]].copy()
            X["hdd2"] = d["hdd"] ** 2
            for k in range(1, 5):
                X[f"dow{k}"] = (d["dow"] == k).astype(float)
            return X.astype(float).values
        s_tr = SARIMAX(train["da_on"].values, exog=exog(train), order=(1, 0, 1), trend="c").fit(disp=False)
        s_all = SARIMAX(df["da_on"].values, exog=exog(df), order=(1, 0, 1), trend="c").filter(s_tr.params)
        preds.loc[test.index, "SARIMAX(1,0,1) + weather/gas"] = s_all.predict()[test.index]
    except Exception as e:
        print("SARIMAX skipped:", e)

    P = preds.loc[test.index].dropna()
    A = y.loc[P.index]
    winter = df.loc[P.index, "month"].isin([12, 1, 2])
    rows = []
    for m in P.columns:
        e = A - P[m]
        rows.append({"Model": m, "MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()),
                     "Bias (actual - pred)": e.mean(), "MAE winter": e[winter].abs().mean(),
                     "MAE rest": e[~winter].abs().mean(), "Worst miss": e.abs().max()})
    table = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True).round(2)
    base = fits["Regression: weather + gas + FE"]
    sens = pd.DataFrame({"Coefficient": base.params, "HAC std error": base.bse,
                         "p-value": base.pvalues}).loc[["hdd", "cdd", "gas"]].round(3)
    return {"table": table, "sens": sens, "preds": P, "actual": A, "df": df, "n_test": len(P)}

In [ ]:
# Step 7: run Zone J and Zone K
results = {}
for z in ["J", "K"]:
    df_z = model_frame(ny, z)
    print(f"\n=== Zone {z}: {len(df_z)} on-peak days "
          f"({df_z.date.min():%Y-%m-%d} to {df_z.date.max():%Y-%m-%d}) ===")
    results[z] = run_models(df_z)
    print(f"Out-of-sample comparison, {results[z]['n_test']} on-peak days in {TEST_START[:4]} ($/MWh)")
    show(results[z]["table"])
    results[z]["table"].to_csv(f"nyiso_zone_{z}_model_comparison.csv", index=False)

if "gas_z6" in ny.columns:
    print("\n=== Zone J with Transco Z6 NY as the gas driver ===")
    results["J_z6"] = run_models(model_frame(ny, "J", gas_col="gas_z6"))
    show(results["J_z6"]["table"].head(5))

In [ ]:
# Step 8: sensitivities (training period, 2023 to 2025)
# Explain: $/MWh change in the on-peak price per extra heating degree day, cooling degree day,
# and per $1/MMBtu of Henry Hub, holding day-of-week and month fixed effects constant.
for z in ["J", "K"]:
    print(f"\nZone {z} sensitivities ($/MWh on-peak):")
    print(results[z]["sens"])

In [ ]:
# Step 9: Zone J minus PJM Western Hub on-peak spread
# Explain: positive spread = NYC priced above PJM West. Reflects NYC's import constraints,
# local generation requirements, and gas basis. Uses the PJM daily file if present; otherwise pulls it.
if os.path.exists(PJM_CACHE):
    pjm = pd.read_csv(PJM_CACHE, parse_dates=["date"])[["date", "da_on"]]
    print("Using PJM daily data from", PJM_CACHE)
else:
    print("pjm_daily.csv not found; pulling PJM Western Hub DA hourly")
    pjm_h = client.get_dataset(dataset="pjm_lmp_day_ahead_hourly", start=START, end=END,
                               filter_column="location", filter_value="WESTERN HUB")
    pjm = daily_blocks(pjm_h[["interval_start_utc", "lmp"]], "da").reset_index()[["date", "da_on"]]

sp = ny[["date", "J_on"]].merge(pjm.rename(columns={"da_on": "pjm_on"}), on="date").dropna()
sp["spread"] = sp["J_on"] - sp["pjm_on"]
sp["year"] = sp["date"].dt.year
sp["month"] = sp["date"].dt.month

by_year = sp.groupby("year").agg(days=("spread", "size"), zone_j=("J_on", "mean"),
                                  pjm_west=("pjm_on", "mean"), avg_spread=("spread", "mean"),
                                  median_spread=("spread", "median")).round(2).reset_index()
by_month = sp.groupby("month").agg(avg_spread=("spread", "mean"), median_spread=("spread", "median"),
                                    days=("spread", "size")).round(2).reset_index()
widest = sp.reindex(sp["spread"].abs().sort_values(ascending=False).index).head(10)
widest = widest.assign(date=widest["date"].dt.strftime("%Y-%m-%d (%a)"))[["date", "J_on", "pjm_on", "spread"]].round(2)

print("Spread by year ($/MWh):"); show(by_year)
print("\nSpread by calendar month, all years ($/MWh):"); show(by_month)
print("\n10 widest spread days:"); show(widest)
for name, t in [("spread_by_year", by_year), ("spread_by_month", by_month), ("spread_widest_days", widest)]:
    t.to_csv(f"nyiso_{name}.csv", index=False)

In [ ]:
# Step 10: largest Zone J forecast misses (best model, 2026 out-of-sample)
# Explain: dates to research (heat waves, cold snaps, transmission or generation outages).
rJ = results["J"]
FUNDAMENTAL = ("Regression", "Heat-rate", "SARIMAX")
best_J = rJ["table"].loc[0, "Model"]                                   # best of all 15
best_fund = rJ["table"][rJ["table"]["Model"].str.startswith(FUNDAMENTAL)].iloc[0]["Model"]  # best weather/gas model
print("Best overall:", best_J, "| Best fundamentals model:", best_fund)
err = (rJ["actual"] - rJ["preds"][best_fund])
top = err.abs().sort_values(ascending=False).head(10).index
misses = rJ["df"].loc[top, ["date", "da_on", "temp", "hdd", "cdd", "gas"]].copy()
misses.insert(2, "forecast", rJ["preds"].loc[top, best_fund].values)
misses.insert(3, "miss", err.loc[top].values)
misses["date"] = misses["date"].dt.strftime("%Y-%m-%d (%a)")
print(f"Top 10 Zone J misses, model: {best_fund}"); show(misses.round(1))
misses.round(1).to_csv("nyiso_zone_J_largest_misses.csv", index=False)

In [ ]:
# Step 11: charts
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
dJ = rJ["df"].loc[rJ["preds"].index, "date"]
axes[0].plot(dJ, rJ["actual"], color="black", lw=1.2, label="Actual Zone J DA on-peak")
axes[0].plot(dJ, rJ["preds"][best_fund], lw=1, label=f"Best fundamentals: {best_fund}")
axes[0].plot(dJ, rJ["preds"]["Naive: previous day"], lw=0.8, alpha=0.6, label="Naive: previous day")
axes[0].set_title(f"NYISO Zone J on-peak DA: actual vs forecast ({TEST_START[:4]}, out-of-sample)")
axes[0].set_ylabel("$/MWh"); axes[0].legend()
m = sp.set_index("date")["spread"].resample("MS").mean()
axes[1].bar(m.index, m.values, width=20)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Zone J minus PJM Western Hub, monthly average on-peak spread")
axes[1].set_ylabel("$/MWh")
plt.tight_layout(); plt.savefig("nyiso_results.png", dpi=150); plt.show()

## New York project-finance modules
The model above answers "what will NYC power cost." These modules answer the project-finance question: **what is a megawatt-hour from a specific New York project worth, and who carries the risk?**

- **Module A, capture price:** price actually earned by Long Island solar and offshore wind, vs. the average price
- **Module B, virtual PPA:** monthly cash flows and risks of a fixed-price PPA on a Long Island solar project
- **Module C, battery arbitrage:** energy-arbitrage revenue of a 4-hour battery in NYC and Long Island

**Assumptions (state these when presenting):** generation profiles are *modeled* from hourly weather (Open-Meteo), not actual plant output; prices are day-ahead; battery uses perfect foresight of day-ahead prices; capacity payments, RECs/ORECs, and ancillary services are excluded.

In [ ]:
# Step 12a: hourly generation profiles from weather
# Explain: solar output scales with sunlight (W/m2); wind output follows a generic turbine power curve.
def get_hourly_weather(lat, lon, variables):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": lat, "longitude": lon, "start_date": START, "end_date": END,
        "hourly": ",".join(variables), "wind_speed_unit": "ms", "timezone": "GMT"}, timeout=120)
    if r.status_code != 200:
        raise StopRun(f"Open-Meteo hourly returned {r.status_code}: {r.text[:200]}")
    js = r.json()["hourly"]
    w = pd.DataFrame({v: js[v] for v in variables})
    w["interval_start_utc"] = pd.to_datetime(js["time"]).tz_localize("UTC")
    return w

PROFILE_CACHE = "ny_profiles.csv"
if os.path.exists(PROFILE_CACHE):
    profiles = pd.read_csv(PROFILE_CACHE, parse_dates=["interval_start_utc"])
    profiles["interval_start_utc"] = pd.to_datetime(profiles["interval_start_utc"], utc=True)
else:
    sol = get_hourly_weather(SOLAR_SITE[1], SOLAR_SITE[2], ["shortwave_radiation"])
    # Open-Meteo radiation is the average over the PRECEDING hour, so it belongs to the interval starting 1 hour earlier
    sol["interval_start_utc"] = sol["interval_start_utc"] - pd.Timedelta(hours=1)
    # 1 MW solar: output = sunlight / 1000 W/m2 x 85% performance ratio, capped at 1 MW
    sol["solar_mw"] = (sol["shortwave_radiation"] / 1000 * 0.85).clip(0, 1)
    wnd = get_hourly_weather(WIND_SITE[1], WIND_SITE[2], ["wind_speed_100m"])
    v = wnd["wind_speed_100m"]
    # Generic offshore turbine, per 1 MW: cut-in 3 m/s, rated 12 m/s, cut-out 25 m/s; 10% wake/availability losses
    cf = ((v**3 - 3**3) / (12**3 - 3**3)).clip(0, 1)
    cf[(v < 3) | (v > 25)] = 0
    wnd["wind_mw"] = cf * 0.90
    profiles = sol[["interval_start_utc", "solar_mw"]].merge(wnd[["interval_start_utc", "wind_mw"]], on="interval_start_utc")
    profiles.to_csv(PROFILE_CACHE, index=False)

hp = ny_hourly.merge(profiles, on="interval_start_utc", how="inner").dropna()
hp["t_et"] = hp["interval_start_utc"].dt.tz_convert("America/New_York")
# Keep only hours from Jan 1 Eastern onward: the first UTC hours of Jan 1 are still Dec 31 in New York
hp = hp[hp["t_et"] >= pd.Timestamp(START, tz="America/New_York")].reset_index(drop=True)
hp["year"] = hp["t_et"].dt.year
hp["month"] = hp["t_et"].dt.to_period("M").astype(str)
print(f"Hourly rows with prices and profiles: {len(hp):,}")
print("Modeled capacity factors:  solar {:.1%}   offshore wind {:.1%}".format(hp["solar_mw"].mean(), hp["wind_mw"].mean()))

In [ ]:
# Step 12b: Module A, capture price
# Explain: capture price = generation-weighted average price. Capture rate = capture price / simple average price.
# Below 100% means the project produces mostly when power is cheap (solar at midday, wind on mild windy nights).
rows = []
for yr, g in hp.groupby("year"):
    for z in ["J", "K"]:
        avg = g[z].mean()
        row = {"Year": yr, "Zone": z, "Avg price (ATC)": avg}
        for tech in ["solar", "wind"]:
            cap = (g[z] * g[f"{tech}_mw"]).sum() / g[f"{tech}_mw"].sum()
            row[f"{tech.title()} capture $/MWh"] = cap
            row[f"{tech.title()} capture rate"] = cap / avg
        rows.append(row)
capture = pd.DataFrame(rows).round(3)
print("Capture prices by year and zone ($/MWh; rate = capture / around-the-clock average):")
show(capture)
capture.to_csv("ny_capture_prices.csv", index=False)

In [ ]:
# Step 12c: Module B, virtual PPA on a 100 MW Long Island solar project
# Explain: a buyer pays the project a fixed strike and receives the market price for every MWh produced.
# Hourly settlement to the buyer = (market price - strike) x generation. Positive = buyer gains.
# Strike is set at the project's 2023 to 2025 average Zone K capture price, i.e. a "fair" price based on history,
# so 2026 shows what happens when the market moves after signing.
MW = 100
hp["gen_mwh"] = hp["solar_mw"] * MW
train_h = hp[hp["t_et"] < pd.Timestamp(TEST_START, tz="America/New_York")]
STRIKE = (train_h["K"] * train_h["gen_mwh"]).sum() / train_h["gen_mwh"].sum()
hp["settle_K"] = (hp["K"] - STRIKE) * hp["gen_mwh"]          # settled at Long Island zone, where the project sits
hp["settle_J"] = (hp["J"] - STRIKE) * hp["gen_mwh"]          # same contract settled at NYC zone (introduces basis)
hp["basis"] = (hp["J"] - hp["K"]) * hp["gen_mwh"]

vppa_year = hp.groupby("year").agg(gen_GWh=("gen_mwh", lambda x: x.sum() / 1000),
                                   capture_K=("K", lambda x: np.nan),
                                   settlement_K_musd=("settle_K", lambda x: x.sum() / 1e6),
                                   settlement_J_musd=("settle_J", lambda x: x.sum() / 1e6),
                                   basis_musd=("basis", lambda x: x.sum() / 1e6)).reset_index()
vppa_year["capture_K"] = hp.groupby("year").apply(lambda g: (g["K"] * g["gen_mwh"]).sum() / g["gen_mwh"].sum()).values
vppa_year = vppa_year.round(2)
print(f"Strike = ${STRIKE:.2f}/MWh (2023-2025 Zone K solar capture price). $ in millions; positive = buyer gains.")
show(vppa_year)

vppa_month = hp.groupby("month").agg(gen_GWh=("gen_mwh", lambda x: x.sum() / 1000),
                                     settlement_K_musd=("settle_K", lambda x: x.sum() / 1e6),
                                     basis_musd=("basis", lambda x: x.sum() / 1e6)).round(2).reset_index()
# Volume risk: how far monthly generation strays from its average for that calendar month
vppa_month["cal_month"] = vppa_month["month"].str[-2:]
vppa_month["gen_vs_normal"] = (vppa_month["gen_GWh"] / vppa_month.groupby("cal_month")["gen_GWh"].transform("mean") - 1).round(3)
print("\nLast 12 months of settlements:")
show(vppa_month.drop(columns="cal_month").tail(12))
vppa_year.to_csv("ny_vppa_by_year.csv", index=False)
vppa_month.drop(columns="cal_month").to_csv("ny_vppa_by_month.csv", index=False)

risk = pd.DataFrame({
    "Risk": ["Shape (solar capture vs. average, Zone K)", "Basis (Zone J minus Zone K, per solar MWh)", "Volume (monthly generation vs. normal)"],
    "Measure": [f"{capture.query('Zone==\'K\'')['Solar capture rate'].mean():.1%} average capture rate",
                f"${(hp['basis'].sum() / hp['gen_mwh'].sum()):+.2f}/MWh average",
                f"{vppa_month['gen_vs_normal'].abs().mean():.1%} average monthly deviation"]})
print("\nPPA risk summary:"); show(risk)

In [ ]:
# Step 12d: Module C, battery arbitrage (4-hour battery, NYC and Long Island)
# Explain: each day, charge in the 4 cheapest day-ahead hours and discharge in the 4 most expensive.
# 85% round-trip efficiency: 4 MWh charged returns 3.4 MWh. Skip the day if the spread doesn't cover losses.
# Perfect foresight of DA prices and one cycle per day: an upper bound on energy-arbitrage revenue.
RTE = 0.85
hp["date_et"] = hp["t_et"].dt.tz_localize(None).dt.normalize()
def daily_arb(x):
    s = np.sort(x.values)
    if len(s) < 8:
        return np.nan
    return max(RTE * s[-4:].sum() - s[:4].sum(), 0.0)    # $ per MW of power capacity per day

batt = pd.DataFrame({z: hp.groupby("date_et")[z].apply(daily_arb) for z in ["J", "K"]})
batt["year"] = batt.index.year
batt_year = batt.groupby("year").agg(J_usd_per_kw_yr=("J", lambda x: x.sum() / 1000),
                                     K_usd_per_kw_yr=("K", lambda x: x.sum() / 1000),
                                     days=("J", "count")).round(2).reset_index()
batt_year["J_annualized"] = (batt_year["J_usd_per_kw_yr"] * 365 / batt_year["days"]).round(2)
batt_year["K_annualized"] = (batt_year["K_usd_per_kw_yr"] * 365 / batt_year["days"]).round(2)
print("4-hour battery energy-arbitrage revenue ($/kW-year; 'annualized' scales partial years to 365 days):")
show(batt_year)
best_days = batt["J"].sort_values(ascending=False).head(10).round(0).reset_index()
best_days.columns = ["date", "Zone J $/MW-day"]
best_days["date"] = best_days["date"].dt.strftime("%Y-%m-%d (%a)")
print("\n10 most valuable battery days in Zone J:"); show(best_days)
batt_year.to_csv("ny_battery_by_year.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cK = capture[capture["Zone"] == "K"]
axes[0].plot(cK["Year"], cK["Solar capture rate"], marker="o", label="Solar, Zone K")
axes[0].plot(cK["Year"], cK["Wind capture rate"], marker="o", label="Offshore wind, Zone K")
axes[0].axhline(1, color="black", lw=0.8); axes[0].set_title("Capture rate (capture / average price)")
axes[0].legend(); axes[0].set_xticks(cK["Year"])
axes[1].bar(batt_year["year"] - 0.2, batt_year["J_annualized"], width=0.4, label="Zone J (NYC)")
axes[1].bar(batt_year["year"] + 0.2, batt_year["K_annualized"], width=0.4, label="Zone K (LI)")
axes[1].set_title("4-hour battery arbitrage, $/kW-year"); axes[1].legend(); axes[1].set_xticks(batt_year["year"])
plt.tight_layout(); plt.savefig("ny_finance_results.png", dpi=150); plt.show()

## More New York modules
- **Module D, utility hedging risk:** a utility must buy 100 MW of NYC power every on-peak hour. What does it cost to buy at the day-ahead price instead of a fixed price locked in a year earlier?
- **Module E, offshore wind PPA:** the solar PPA analysis applied to a 100 MW offshore wind farm delivering into NYC (Zone J)
- **Module F, forecast ranges:** an 80% price range around each Zone J forecast, and how often 2026 prices actually fell inside it

In [ ]:
# Step 12e: Module D, utility hedging risk (100 MW on-peak block, Zone J)
# Explain: 100 MW x 16 on-peak hours = 1,600 MWh per weekday. "Hedged" price for each year = the previous year's
# average on-peak price, a simple proxy for a fixed price bought a year ahead (actual forward prices not used).
# Daily overrun = (day-ahead price - hedged price) x 1,600 MWh. Positive = buying at market cost more than the hedge.
BLOCK_MWH = 100 * 16
dJ_on = ny[["date", "J_on"]].dropna().copy()
dJ_on["year"] = dJ_on["date"].dt.year
hedge_px = dJ_on.groupby("year")["J_on"].mean().shift(1)          # prior-year average
dJ_on["hedge_px"] = dJ_on["year"].map(hedge_px)
dJ_on = dJ_on.dropna(subset=["hedge_px"])
dJ_on["overrun"] = (dJ_on["J_on"] - dJ_on["hedge_px"]) * BLOCK_MWH
dJ_on["month"] = dJ_on["date"].dt.to_period("M")
monthly_over = dJ_on.groupby("month")["overrun"].sum()
hedge = dJ_on.groupby("year").agg(
    hedge_price=("hedge_px", "first"), avg_market=("J_on", "mean"),
    market_cost_musd=("J_on", lambda x: (x * BLOCK_MWH).sum() / 1e6),
    overrun_musd=("overrun", lambda x: x.sum() / 1e6),
    var95_day_usd=("overrun", lambda x: x.quantile(0.95)),
    worst_day_usd=("overrun", "max")).reset_index()
hedge["worst_month_musd"] = hedge["year"].map(
    monthly_over.groupby(monthly_over.index.year).max() / 1e6)
hedge = hedge.round(2)
print("Utility buying 100 MW of NYC on-peak power: market vs. prior-year fixed price")
print("var95_day_usd = on 1 day in 20, the daily overrun is at least this large\n")
show(hedge)
hedge.to_csv("ny_utility_hedging.csv", index=False)

In [ ]:
# Step 12f: Module E, offshore wind PPA (100 MW, settled at Zone J) vs. the solar PPA (settled at Zone K)
# Explain: same contract as Module B. Strike = the project's own 2023 to 2025 capture price at its settlement zone.
def vppa_table(gen_col, zone, label):
    g = hp.copy()
    g["gen"] = g[gen_col] * 100
    tr_mask = g["t_et"] < pd.Timestamp(TEST_START, tz="America/New_York")
    strike = (g.loc[tr_mask, zone] * g.loc[tr_mask, "gen"]).sum() / g.loc[tr_mask, "gen"].sum()
    g["settle"] = (g[zone] - strike) * g["gen"]
    t = g.groupby("year").apply(lambda x: pd.Series({
        "gen_GWh": x["gen"].sum() / 1000,
        "capture": (x[zone] * x["gen"]).sum() / x["gen"].sum(),
        "buyer_settlement_musd": x["settle"].sum() / 1e6})).reset_index()
    t.insert(0, "project", label); t["strike"] = strike
    return t.round(2)

ppa_cmp = pd.concat([vppa_table("solar_mw", "K", "100 MW Long Island solar (Zone K)"),
                     vppa_table("wind_mw", "J", "100 MW offshore wind into NYC (Zone J)")])
print("Fixed-price PPA comparison ($M; positive = buyer gains, project gave up that upside):\n")
show(ppa_cmp)
ppa_cmp.to_csv("ny_ppa_comparison.csv", index=False)

In [ ]:
# Step 12g: Module F, forecast ranges for Zone J
# Explain: refit the best regression on 2023 to 2025, take the 10th and 90th percentile of its training errors
# (separately for winter and the rest of the year), and add them to each 2026 forecast. A well-calibrated
# 80% range should contain the actual price about 80% of the time.
f_best = FORMULAS["Regression: + squared weather + lag"]
dfJ = rJ["df"]
trJ = dfJ[dfJ["date"] < TEST_START]
fitJ = smf.ols(f_best, data=trJ).fit()
res = trJ["da_on"] - fitJ.fittedvalues
season = lambda d: np.where(d["month"].isin([12, 1, 2]), "Winter (Dec-Feb)", "Rest of year")
bands = pd.DataFrame({"res": res, "season": season(trJ)}).groupby("season")["res"].quantile([0.10, 0.90]).unstack()
teJ = dfJ[dfJ["date"] >= TEST_START].copy()
teJ["fc"] = fitJ.predict(teJ)
teJ["season"] = season(teJ)
teJ["lo"] = teJ["fc"] + teJ["season"].map(bands[0.10])
teJ["hi"] = teJ["fc"] + teJ["season"].map(bands[0.90])
teJ["inside"] = teJ["da_on"].between(teJ["lo"], teJ["hi"])
cover = teJ.groupby("season").agg(days=("inside", "size"), share_inside=("inside", "mean"),
                                  avg_range_width=("hi", lambda x: (x - teJ.loc[x.index, "lo"]).mean())).reset_index()
cover.loc[len(cover)] = ["All 2026", len(teJ), teJ["inside"].mean(), (teJ["hi"] - teJ["lo"]).mean()]
cover["share_inside"] = (cover["share_inside"] * 100).round(1)
cover["avg_range_width"] = cover["avg_range_width"].round(2)
print("Zone J 80% forecast ranges, 2026 (target: about 80% of days inside the range)\n")
show(cover.rename(columns={"share_inside": "% of days inside", "avg_range_width": "avg range width $/MWh"}))
cover.to_csv("ny_forecast_ranges.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(teJ["date"], teJ["lo"], teJ["hi"], alpha=0.3, label="80% forecast range")
ax.plot(teJ["date"], teJ["fc"], lw=1, label="Forecast")
ax.plot(teJ["date"], teJ["da_on"], color="black", lw=1, label="Actual")
ax.set_title("Zone J on-peak: forecast with 80% range, 2026"); ax.set_ylabel("$/MWh"); ax.legend()
plt.tight_layout(); plt.savefig("ny_forecast_ranges.png", dpi=150); plt.show()

In [ ]:
# Step 12: README section and resume line, filled from the results above
tJ = rJ["table"].set_index("Model")
mae_best = tJ.loc[best_fund, "MAE"]
mae_naive = tJ.loc["Naive: previous day", "MAE"]
beats_naive = mae_best < mae_naive
yr_last = by_year.iloc[-1]
widest_month = by_month.loc[by_month["avg_spread"].abs().idxmax()]
month_name = pd.Timestamp(2000, int(widest_month["month"]), 1).strftime("%B")
pct = (1 - mae_best / mae_naive) * 100
finding = (f"Zone J averaged ${yr_last['avg_spread']:+.2f}/MWh vs. PJM West in {int(yr_last['year'])}; "
           f"widest in {month_name} (avg ${widest_month['avg_spread']:+.2f}/MWh)")

readme = f"""## NYISO extension: Zone J (NYC) and Zone K (Long Island)

**Question:** do the weather-and-gas fundamentals that explain PJM Western Hub also explain New York City and Long Island on-peak power, and how does NYC price relative to PJM?

**Data:** NYISO hourly day-ahead zonal LBMPs from the Grid Status API (dataset `{DATASET}`, zones `{ZONE_NAMES['J']}` and `{ZONE_NAMES['K']}`), {START} to {END}; daily mean temperature for New York City (Zone J) and Islip (Zone K) from Open-Meteo, converted to degree days (base 65F); Henry Hub spot from EIA, previous trading day.

**Method:** identical to the PJM build: on-peak blocks (weekdays, HE8 to HE23 Eastern, NERC holidays excluded), chronological split (train 2023 to 2025, test 2026), 15-model out-of-sample comparison, HAC standard errors.

**Results (2026 out-of-sample, Zone J):** best weather/gas model {best_fund}, MAE ${mae_best:.2f}/MWh vs. ${mae_naive:.2f}/MWh for the naive previous-day benchmark ({abs(pct):.0f}% {"lower" if beats_naive else "higher"}). Best of all 15 models: {best_J} (MAE ${tJ.loc[best_J, 'MAE']:.2f}/MWh). Zone K best of all 15: {results['K']['table'].loc[0, 'Model']} (MAE ${results['K']['table'].loc[0, 'MAE']:.2f}/MWh).

**Zone J vs. PJM West:** {finding}.

**Project-finance modules (modeled profiles, day-ahead prices):**
- Capture rate, latest year: Zone K solar {cK.iloc[-1]['Solar capture rate']:.0%}, Zone K offshore wind {cK.iloc[-1]['Wind capture rate']:.0%}; Zone J solar {capture[capture.Zone=='J'].iloc[-1]['Solar capture rate']:.0%}.
- Virtual PPA (100 MW Long Island solar, strike ${STRIKE:.2f}/MWh = 2023-2025 capture price): {int(vppa_year.iloc[-1]['year'])} buyer settlement ${vppa_year.iloc[-1]['settlement_K_musd']:+.2f}M at Zone K; Zone J settlement would add ${vppa_year.iloc[-1]['basis_musd']:+.2f}M of basis.
- 4-hour battery arbitrage (annualized $/kW-year): Zone J {', '.join(f"{int(r.year)}: ${r.J_annualized:.0f}" for r in batt_year.itertuples())}.
- Utility hedging (100 MW NYC on-peak, hedged at prior-year average): {', '.join(f"{int(r.year)}: overrun ${r.overrun_musd:+.1f}M, 1-in-20 day ${r.var95_day_usd:,.0f}" for r in hedge.itertuples())}.
- Offshore wind PPA (100 MW, Zone J, strike ${ppa_cmp.iloc[-1]['strike']:.2f}/MWh): {int(ppa_cmp.iloc[-1]['year'])} buyer settlement ${ppa_cmp.iloc[-1]['buyer_settlement_musd']:+.2f}M.
- Zone J 80% forecast range held {cover.iloc[-1]['share_inside']:.0f}% of 2026 days (average width ${cover.iloc[-1]['avg_range_width']:.0f}/MWh).

**Limitations:** generation profiles are modeled from weather, not metered output; capacity payments, RECs/ORECs and ancillary services excluded; battery assumes perfect day-ahead foresight. Henry Hub understates NYC gas cost in winter; Transco Zone 6 NY, the relevant local price, has no free daily series. Backtest uses observed weather (perfect-forecast assumption). No transmission, outage, or load-forecast variables yet.
"""
with open("README_NYISO_section.md", "w") as f:
    f.write(readme)
print(readme)
print("RESUME LINE (verify every number against the tables above):")
if beats_naive:
    print(f"Extended the model to NYISO Zone J (NYC): ${mae_best:.2f}/MWh MAE vs. ${mae_naive:.2f}/MWh naive; {finding}")
else:
    print(f"NOTE: the best weather/gas model ({best_fund}, ${mae_best:.2f}/MWh) did NOT beat naive (${mae_naive:.2f}/MWh) in Zone J. "
          "Do not use the standard resume line; report the finding instead, e.g.:")
    print(f"Extended the model to NYISO Zone J (NYC): price persistence beat weather/gas fundamentals out-of-sample; {finding}")
print("\nFINANCE RESUME LINE (verify against the tables above):")
print(f"Modeled New York project revenue: Long Island solar captured {cK.iloc[-1]['Solar capture rate']:.0%} of the average Zone K price in "
      f"{int(cK.iloc[-1]['Year'])}; 4-hour NYC battery arbitrage worth ${batt_year.iloc[-1]['J_annualized']:.0f}/kW-year; "
      f"priced shape and basis risk on a 100 MW solar virtual PPA")